In [1]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ninhgiangnguyen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ninhgiangnguyen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/ninhgiangnguyen/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from scipy import stats
import seaborn as sns

import os 
import sys

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.show()
plt.rcParams.update({'font.size': 18})

np.set_printoptions(suppress=True, precision=3)

In [5]:
import torch
print(torch.cuda.is_available())

False


In [6]:
from bertopic import BERTopic

# 1. Overal exploration

In [4]:
root_dir = os.path.dirname(os.getcwd())

text_path = os.path.join(root_dir, "data", "review_text_analytics.csv")
df = pd.read_csv(text_path)

In [5]:
df

,REVIEW_KEY,REVIEW_ID,REVIEW_TEXT,AVERAGE_RATING,RATING_BAND,RECOMMENDED,SEAT_TYPE,TYPE_OF_TRAVELLER,HAS_LAYOVER,IS_VERIFIED,...,INFLIGHT_ENTERTAINMENT,GROUND_SERVICE,WIFI_AND_CONNECTIVITY,VALUE_FOR_MONEY,NATIONALITY,NUMBER_OF_FLIGHTS,CAL_YEAR,CAL_QUARTER,CAL_MONTH,AIRLINE_NAME_CLEANED
0,0be42b62f4cffba1d1465e29f98a21f5,148929,The first flight of my year went well with Del...,4.33,good,True,Premium Economy,Family Leisure,False,True,...,NaN,5.0,5.0,4,South Korea,21,2026,1,1,delta air lines
1,0c6e42257d2a045e858315003d40b7b3,149498,I’m at Maui airport right now and my flight fr...,2.14,medium,False,Economy Class,Family Leisure,False,True,...,3.0,1.0,1.0,3,United States,2,2026,1,3,delta air lines
2,2b2554f40e8617dbe373bb5103022ee4,149400,"I booked a flight, rental car and hotel with D...",2.00,medium,False,Economy Class,Business,False,True,...,NaN,3.0,NaN,1,United States,1,2026,1,3,delta air lines
3,4238f2b19a845f2235b950299d833e81,149497,I’m at Maui airport right now and my flight fr...,2.14,medium,False,Economy Class,Family Leisure,False,True,...,3.0,1.0,1.0,3,United States,2,2026,1,3,delta air lines
4,5023ccaa3433b4d7655c431359b991d5,149049,I purchased an international itinerary directl...,1.00,bad,False,Economy Class,Family Leisure,False,True,...,NaN,NaN,NaN,1,United States,1,2026,1,1,delta air lines
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1389,e1ff6abde692b6dbcb0c43ffbeadd597,70441,My husband and I took a trip to celebrate our ...,5.00,good,True,Economy Class,Couple Leisure,False,True,...,5.0,5.0,NaN,5,United States,6,2017,4,12,delta air lines
1390,f4c41f38d393461b79d46889a95bdeac,72191,Los Angeles to New York JFK. Delta One is dece...,3.57,medium,True,Business Class,Solo Leisure,False,True,...,3.0,4.0,2.0,3,United States,5,2017,4,11,delta air lines
1391,f89682e9d1e9724cda9ced1ab15e6ad1,70564,"New York to Milan. The flight was good, pushba...",2.33,medium,True,Economy Class,Family Leisure,False,True,...,3.0,3.0,NaN,2,United Arab Emirates,1,2017,3,8,delta air lines
1392,fba8fdfb072aa3dcd44db083b0b24c4a,70241,This was a family holiday to Disney. Myself an...,2.00,medium,False,Economy Class,Family Leisure,True,True,...,2.0,2.0,NaN,3,United Kingdom,2,2017,4,11,delta air lines


In [6]:
df['REVIEW_TEXT']

0       The first flight of my year went well with Del...
1       I’m at Maui airport right now and my flight fr...
2       I booked a flight, rental car and hotel with D...
3       I’m at Maui airport right now and my flight fr...
4       I purchased an international itinerary directl...
                              ...                        
1389    My husband and I took a trip to celebrate our ...
1390    Los Angeles to New York JFK. Delta One is dece...
1391    New York to Milan. The flight was good, pushba...
1392    This was a family holiday to Disney. Myself an...
1393    On December 29th 2016 I flew Delta business Cl...
Name: REVIEW_TEXT, Length: 1394, dtype: str

In [7]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

# 2. Preprocessing data 

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

In [9]:
df['clean_text'] = df['REVIEW_TEXT'].apply(clean_text)

In [10]:
df['clean_text'] 

0       first flight year went well delta connection s...
1       im maui airport right flight delta got cancele...
2       booked flight rental car hotel delta hotel air...
3       im maui airport right flight delta got cancele...
4       purchased international itinerary directly del...
                              ...                        
1389    husband took trip celebrate year anniversary f...
1390    los angeles new york jfk delta one decent tran...
1391    new york milan flight good pushback expected t...
1392    family holiday disney wife son wife three gran...
1393    december th flew delta business class seattle ...
Name: clean_text, Length: 1394, dtype: str

# 3. Modeling text data

In [1]:


# Disable probabilities to stop the 3-hour hang
topic_model = BERTopic(calculate_probabilities=False, verbose=True)

topics, probs = topic_model.fit_transform(df['clean_text'])

KeyboardInterrupt: 

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,559,-1_flight_delta_hour_time,"[flight, delta, hour, time, get, seat, airline...",[first time flown delta family flew elmira ny ...
1,0,272,0_flight_seat_delta_class,"[flight, seat, delta, class, good, service, cr...",[booked first class trip two delta airline usi...
2,1,231,1_flight_delta_hour_delayed,"[flight, delta, hour, delayed, time, delay, pl...",[flight delayed time canceled reason pilot run...
3,2,95,2_bag_check_delta_luggage,"[bag, check, delta, luggage, baggage, claim, t...",[new york atlanta delta lost bag way new york ...
4,3,59,3_rude_attendant_mask_flight,"[rude, attendant, mask, flight, delta, custome...",[far worst experience delta head flight attend...
5,4,47,4_flight_delta_get_airport,"[flight, delta, get, airport, time, plane, hou...",[booked flight delta leave chicago phoenix con...
6,5,45,5_worst_flight_delta_hour,"[worst, flight, delta, hour, airline, ever, ex...",[flying delta airline worst travel experience ...
7,6,34,6_customer_service_call_delta,"[customer, service, call, delta, hour, agent, ...",[one worst airline worst customer service arri...
8,7,34,7_seat_family_child_flight,"[seat, family, child, flight, delta, old, kid,...",[worst experience ever family bought ticket mo...
9,8,18,8_seat_row_husband_flight,"[seat, row, husband, flight, agent, delta, wif...",[family traveled ft myers atlanta round trip d...


In [ ]:
topic_model.visualize_topics()

[autoreload of IPython.utils.PyColorize failed: Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    return
           
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    self.imports_froms[module_name] = []
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    continue
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    func_attrs = [
               ^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    continue
  File "/L

Unexpected exception formatting exception. Falling back to standard exception


[autoreload of nbformat.v3 failed: Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    return
           
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/importlib/__init__.py", line 121, in reload
    raise ImportError(f"parent {parent_name!r} not in sys.modules",
ImportError: parent 'nbformat' not in sys.modules
]
[autoreload of nbformat._imports failed: Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    return
           
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-pac